# CSoT'26 — ML in Astronomy — Week 1 · Part 1: Foundations (Solution)

Reference implementation for the Week 1 task. **Only open after attempting [`week1_starter.ipynb`](week1_starter.ipynb).**

The structure mirrors the starter so you can diff your own work against it cell by cell. Each cell has short commentary explaining *why* the code is what it is.

## Step 1 — Confirm the GPU at the OS level

`!nvidia-smi` runs in the underlying Linux VM. If the runtime is on GPU, it prints a table; otherwise the command isn't even installed.

In [ ]:
!nvidia-smi

## Step 2 — Imports and version sanity check

In [ ]:
import torch
import torchvision
import matplotlib
import matplotlib.pyplot as plt

print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")

## Step 3 — Confirm the GPU from PyTorch

`torch.cuda.is_available()` is the authoritative source of truth. `nvidia-smi` showing a GPU is necessary but not sufficient (PyTorch could be built without CUDA support, though that's extremely rare on Colab).

In [ ]:
print("CUDA available :", torch.cuda.is_available())
print("Device count   :", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name    :", torch.cuda.get_device_name(0))

## Step 4 — Define `device`

This idiom keeps the rest of the notebook portable: it'll run as-is on a CPU-only machine, just slower.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

## Step 5 — Toy galaxy tensor on CPU

Shape `(3, 64, 64)` matches one resized RGB galaxy image: 3 channels × 64 × 64 pixels.
Default `dtype` is `torch.float32` (single precision, the deep-learning standard).

In [ ]:
x = torch.randn(3, 64, 64)
print("shape  :", x.shape)
print("dtype  :", x.dtype)
print("device :", x.device)

## Step 6 — Move it to the GPU

`.to(device)` returns a *new* tensor when moving across devices; `x` itself stays on CPU. The shape and dtype are preserved.

In [ ]:
x_gpu = x.to(device)
print("device after move :", x_gpu.device)
print("shape  :", x_gpu.shape)
print("dtype  :", x_gpu.dtype)
print("original x still on:", x.device)

## Step 7 — A matmul on the GPU

`x_gpu[0]` slices out the first channel — a `(64, 64)` matrix. The `@` operator does matrix multiplication. The result stays on the same device as its inputs.

In [ ]:
result = x_gpu[0] @ x_gpu[0]
print("result shape  :", result.shape)
print("result device :", result.device)

## Stretch Goal 1 — CPU vs GPU benchmark

Two things to notice:

1. The **warm-up** runs before timing — the first GPU call includes one-time initialisation overhead and would skew the result.
2. The `torch.cuda.synchronize()` calls — GPU work is asynchronous; we must wait for it to actually finish before stopping the clock.

In [ ]:
import time

SIZE = 4096
N_RUNS = 5

def bench(dev):
    a = torch.randn(SIZE, SIZE, device=dev)
    b = torch.randn(SIZE, SIZE, device=dev)
    for _ in range(2):
        (a @ b).sum().item()
    if dev == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        c = a @ b
    if dev == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) / N_RUNS

cpu_t = bench("cpu")
print(f"CPU avg : {cpu_t*1000:7.1f} ms")
if torch.cuda.is_available():
    gpu_t = bench("cuda")
    print(f"GPU avg : {gpu_t*1000:7.1f} ms")
    print(f"Speedup : {cpu_t / gpu_t:.1f}x")

## Stretch Goal 2 — Visualise a 'synthetic galaxy'

This is not real astrophysics — it's a cartoon. But:

- The exponential `exp(-r * 4)` term mimics the way real galaxies' surface brightness *decays* with radius. Week 2 will formalise this as the **Sérsic profile**.
- The `cos(2θ + 8r)` term gives us a tidy 2-arm logarithmic spiral. Real spiral arms are density waves, not rigid rotators — Week 3 territory.

Don't read physics into the colormap; it's purely aesthetic.

In [ ]:
import torch
import matplotlib.pyplot as plt

H, W = 128, 128
y, x_grid = torch.meshgrid(
    torch.linspace(-1, 1, H),
    torch.linspace(-1, 1, W),
    indexing="ij",
)
r = torch.sqrt(x_grid**2 + y**2)
theta = torch.atan2(y, x_grid)

disk = torch.exp(-r * 4)
arms = 0.5 * torch.exp(-r * 4) * (1 + torch.cos(2 * theta + 8 * r))
galaxy = (disk + arms).clamp(0, 1).numpy()

plt.figure(figsize=(4, 4))
plt.imshow(galaxy, cmap="inferno")
plt.axis("off")
plt.title("Synthetic 'galaxy' (cartoon, not physics)")
plt.show()

## Reflection (example answers)

Yours will differ — these are just illustrative.

1. **Setup pain.** Forgot to switch the runtime to GPU at first, so `torch.cuda.is_available()` returned `False`. The fix was the menu toggle and a runtime restart.
2. **What a CNN needs to spot a spiral.** Localised features over multiple scales: a bright central bulge, two or more curved arms with a measurable pitch, possibly dust lanes silhouetted against the disk. Edge-on spirals also have a characteristic thin lens shape with a central bulge.
3. **Why a whole week on data pipelines?** Because in production ML, getting data *correctly* loaded, normalised, augmented, and batched is at least as much work as the model itself — and any bug here silently destroys model quality without throwing errors. Better to get it right before introducing a second moving part.

---

Next stop: **Week 1 · Part 2 — PyTorch Data Pipelines** (`week1_data_starter.ipynb`). We'll meet `Dataset`, `DataLoader`, `transforms`, and the Galaxy Zoo 2 images themselves.